# Breast Cancer Classification with Machine Learning

University Introduction to AI project using the Wisconsin Diagnostic Breast Cancer dataset. The goal is to classify tumors as benign or malignant and compare classical machine-learning approaches.


In [ ]:
%pip install -q ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_curve, auc)


## 1. Data loading and preprocessing

The UCI dataset contains 569 samples and 30 numerical features derived from digitized images of fine-needle aspirates. The target is encoded as malignant = 1 and benign = 0.


In [ ]:
wdbc = fetch_ucirepo(id=17)
X = wdbc.data.features.copy()
y = wdbc.data.targets.copy()

desired_feature_names = [
    'mean_radius','mean_texture','mean_perimeter','mean_area','mean_smoothness',
    'mean_compactness','mean_concavity','mean_concave_points','mean_symmetry','mean_fractal_dimension',
    'se_radius','se_texture','se_perimeter','se_area','se_smoothness',
    'se_compactness','se_concavity','se_concave_points','se_symmetry','se_fractal_dimension',
    'worst_radius','worst_texture','worst_perimeter','worst_area','worst_smoothness',
    'worst_compactness','worst_concavity','worst_concave_points','worst_symmetry','worst_fractal_dimension'
]
X.columns = desired_feature_names
y = y.iloc[:, 0].map({'M': 1, 'B': 0})

print('Shape:', X.shape)
print('Missing values:', int(X.isna().sum().sum()))
print(y.value_counts().rename(index={0:'Benign',1:'Malignant'}))


## 2. Exploratory data analysis


In [ ]:
df = X.copy()
df['Diagnosis'] = y

plt.figure(figsize=(6,4))
sns.countplot(x='Diagnosis', data=df)
plt.xticks([0,1], ['Benign','Malignant'])
plt.title('Diagnosis Distribution')
plt.show()

corr = df.corr(numeric_only=True)['Diagnosis'].drop('Diagnosis').sort_values(ascending=False)
plt.figure(figsize=(8,8))
sns.barplot(x=corr.values, y=corr.index)
plt.title('Feature Correlation with Diagnosis')
plt.show()


## 3. Train/test split and scaling


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Baseline model comparison


In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=2, random_state=42),
    'Support Vector Machine': SVC(kernel='linear', C=1.0),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

rows = []
for name, model in models.items():
    train_X = X_train_scaled if name != 'Random Forest' else X_train
    test_X = X_test_scaled if name != 'Random Forest' else X_test
    model.fit(train_X, y_train)
    pred = model.predict(test_X)
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred)
    })

pd.DataFrame(rows).set_index('Model').round(4)


## 5. Hyperparameter tuning

Grid search is used to tune Random Forest and SVM. Recall is emphasized because missing malignant cases is especially costly.


In [ ]:
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 5, 10],
        'criterion': ['gini', 'entropy']
    },
    cv=5, scoring='recall', n_jobs=-1
)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

svm_grid = GridSearchCV(
    SVC(probability=True, random_state=42),
    {
        'C': [0.1, 1, 10, 100],
        'gamma': [1, 0.1, 0.01, 0.001],
        'kernel': ['rbf', 'linear']
    },
    cv=5, scoring='recall', n_jobs=-1
)
svm_grid.fit(X_train_scaled, y_train)
best_svm = svm_grid.best_estimator_

print('Best RF:', rf_grid.best_params_)
print('Best SVM:', svm_grid.best_params_)


## 6. PCA comparison


In [ ]:
pca = PCA(n_components=10, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

rf_pca = RandomForestClassifier(**rf_grid.best_params_, random_state=42)
rf_pca.fit(X_train_pca, y_train)

svm_pca = SVC(**svm_grid.best_params_, probability=True, random_state=42)
svm_pca.fit(X_train_pca, y_train)

print('Original features:', X_train.shape[1])
print('PCA components:', X_train_pca.shape[1])


## 7. Final evaluation


In [ ]:
candidates = {
    'Tuned Random Forest (Full)': (best_rf, X_test),
    'Tuned SVM (Full)': (best_svm, X_test_scaled),
    'Random Forest + PCA': (rf_pca, X_test_pca),
    'SVM + PCA': (svm_pca, X_test_pca)
}

for name, (model, test_X) in candidates.items():
    pred = model.predict(test_X)
    print(f'\n=== {name} ===')
    print(classification_report(y_test, pred, digits=3))
    print(confusion_matrix(y_test, pred))


## 8. Best-model interpretation and ROC curve


In [ ]:
feature_importance = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
feature_importance.head(10).sort_values().plot(kind='barh', figsize=(8,5), title='Top 10 Random Forest Features')
plt.show()

y_prob = best_rf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC={roc_auc:.2f})')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## Conclusion

The project compared Decision Tree, SVM, and Random Forest classifiers, tuned the strongest candidates with cross-validation, and tested PCA-based dimensionality reduction. In the original coursework run, the tuned Random Forest achieved about **97% test accuracy** with **93% recall for malignant cases**, making it the strongest overall model among the tested configurations.
